# DramaFlow Recap backend (Colab)
Video Recap / Scene Recap "Auto" အတွက်။ Altare မရရင် ဒါကို သုံးပါ။
1. Cell 1 ကို run (တစ်ကြိမ်ပဲ၊ ၁ မိနစ်လောက်)
2. Cell 2 ကို run → ထွက်လာတဲ့ `https://….trycloudflare.com#TOKEN` တစ်ကြောင်းလုံးကို DramaFlow ရဲ့ Auto card > Backend ကွက်မှာ ထည့်
3. Colab ကို ပိတ်ရင်/ရပ်ရင် URL ပြောင်းမယ်၊ Cell 2 ကို ပြန် run ပြီး အသစ်ယူပါ


In [ ]:
!pip -q install fastapi uvicorn edge-tts google-genai imageio-ffmpeg yt-dlp
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
!rm -rf /content/dramaflow && git clone -q --depth 1 --branch main https://github.com/punk123321/dramaflow-ai-studio /content/dramaflow
print("ok")

In [ ]:
import os, re, secrets, subprocess, time

TOKEN = secrets.token_urlsafe(18)
env = dict(os.environ, APP_TOKEN=TOKEN)
server = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--log-level", "warning"],
    cwd="/content/dramaflow/tools/dramaflow_backend", env=env,
)
time.sleep(4)
if server.poll() is not None:
    raise SystemExit("Server မတက်ဘူး — အပေါ်က error ကို ကြည့်ပါ")

proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print("Public URL ခဏစောင့်ပါ ...")
for line in proc.stdout:
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\nApp ရဲ့ Backend ကွက်မှာ ဒီတစ်ကြောင်းလုံး ကူးထည့်ပါ:")
        print(f"{m.group(0)}#{TOKEN}")
        break
for _ in proc.stdout:  # keep the cell (and the tunnel) alive
    pass
